#### Implementing PCY Algorithm on Groceries Dataset

In [46]:
import pandas as pd
from collections import defaultdict
import itertools

In [47]:
data = pd.read_csv("data/groceries.csv")

In [48]:
transactions = data.groupby('Member_number')['itemDescription'].apply(list).tolist()

In [49]:
item_counts = defaultdict(int)
for transaction in transactions:
    for item in transaction:
        item_counts[item] += 1

In [50]:
min_support = 2
frequent_items = {item for item, count in item_counts.items() if count >= min_support}

In [51]:
hash_table = defaultdict(list)
pair_counts = defaultdict(int)

In [52]:
bucket_size = 100
def hash_pair(pair):
    return hash(pair) % bucket_size

In [53]:
for transaction in transactions:
    # Filter the transaction to include only frequent items
    filtered_transaction = [item for item in transaction if item in frequent_items]

    # Generate pairs from the filtered transaction
    for pair in itertools.combinations(filtered_transaction, 2):
        bucket = hash_pair(pair)
        hash_table[bucket].append(pair)
        pair_counts[pair] += 1

In [54]:
# Compute the bit vector dynamically
bit_vector = {
    bucket: 1 if sum(1 for pair in pairs if pair_support[pair] >= min_support) > 0 else 0
    for bucket, pairs in hash_table.items()
}

bucket_info = []
for bucket, pairs in hash_table.items():
    pair_support = defaultdict(int)

    # Count the support for each pair in the bucket
    for pair in pairs:
        pair_support[pair] += 1

    # Find the highest support count in the bucket
    highest_support = max(pair_support.values(), default=0)

    # Add bucket information
    bucket_info.append({
        "Bit Vector": bit_vector[bucket],  # Use the computed bit vector
        "Bucket No.": bucket,
        "Highest Support Count": highest_support,  # The highest support count in the bucket
        "Pairs": pairs,  # All raw pairs in the bucket
        "Candidate Set": [pair for pair in pairs if bit_vector[bucket] == 1 and pair_support[pair] >= min_support]  # Filtered candidate pairs
    })

In [55]:
bucket_df = pd.DataFrame(bucket_info)
bucket_df

,Bit Vector,Bucket No.,Highest Support Count,Pairs,Candidate Set
0,0,34,296,"[(soda, canned beer), (rolls/buns, shopping ba...",[]
1,0,95,162,"[(soda, sausage), (soda, sausage), (soda, misc...",[]
2,0,19,411,"[(soda, whole milk), (soda, whole milk), (soda...",[]
3,0,53,295,"[(soda, pickled vegetables), (frankfurter, bee...",[]
4,0,5,236,"[(soda, semi-finished bread), (hamburger meat,...",[]
...,...,...,...,...,...
95,0,54,423,"[(tropical fruit, dessert), (liver loaf, root ...",[]
96,0,40,241,"[(candles, other vegetables), (citrus fruit, c...",[]
97,0,96,243,"[(processed cheese, yogurt), (mustard, whipped...",[]
98,0,66,225,"[(meat, domestic eggs), (sausage, butter milk)...",[]


In [ ]:
# check how many pairs have bit vector 1s and 0s
bucket_df['Bit Vector'].value_counts()

Bit Vector
0    99
1     1
Name: count, dtype: int64

In [57]:
# check how many pairs have candidate sets
bucket_df['Candidate Set'].apply(len).value_counts()

Candidate Set
0       99
1020     1
Name: count, dtype: int64